# Homework 1
In the first week's homework, the objective is to fit a linear regression model to predict the duration of a New York Yellow Taxi ride. For the homework, we use trip records of yellow taxis from January and February 2023.

We begin by importing the required packages and the data.

In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [2]:
df_yellow_jan = pd.read_parquet("../data/yellow_taxi/yellow_tripdata_2023-01.parquet")
df_yellow_feb = pd.read_parquet("../data/yellow_taxi/yellow_tripdata_2023-02.parquet")

## Question 1: Columns in January (D)
The data for January 2023 contains 19 columns.

In [3]:
len(df_yellow_jan.columns)

19

## Question 2: Standard deviation of trip duration (B)
For the data belonging to January 2023, the standard deviation of trip durations is 42.59 minutes.

In [4]:
def process_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Processes the taxi trip dataframe by:
    - Converting pickup and dropoff datetime columns to datetime objects
    - Calculating trip duration in minutes and adding it as a new column
    - Filtering out trips with duration less than 1 minute or greater than 60 minutes

    Args:
        df (pd.DataFrame): Input dataframe with taxi trip records

    Returns:
        pd.DataFrame: Processed dataframe with a 'duration' column and outliers removed
    """
    df.loc[:, "tpep_pickup_datetime"] = pd.to_datetime(
        df["tpep_pickup_datetime"], format="%Y-%m-%d %H:%M:%S"
    )
    df.loc[:, "tpep_dropoff_datetime"] = pd.to_datetime(
        df["tpep_dropoff_datetime"], format="%Y-%m-%d %H:%M:%S"
    )
    df.loc[:, "duration"] = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    df.loc[:, "duration"] = df["duration"].dt.total_seconds() / 60.0

    return df


In [5]:
df_yellow_jan_processed = process_data(df_yellow_jan)
df_yellow_jan_processed["duration"].std()

/tmp/ipykernel_1812/4228674976.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 8.43333333  6.31666667 12.75       ... 24.51666667 13.
 14.4       ]' has dtype incompatible with timedelta64[us], please explicitly cast to a compatible dtype first.
  df.loc[:, "duration"] = df["duration"].dt.total_seconds() / 60.0


np.float64(42.59435124195457)

## Question 3: Dropping outliers (D)
After dropping the outliers, 98% of the records are retained.

In [6]:
orig_jan_count = len(df_yellow_jan)
df_yellow_jan_processed = df_yellow_jan_processed[
    (df_yellow_jan_processed["duration"] >= 1)
    & (df_yellow_jan_processed["duration"] <= 60)
]
filtered_jan_count = len(df_yellow_jan_processed)
filtered_jan_count / orig_jan_count

0.9812202822125979

## Question 4: One-hot encoding (C)
After one-hot encoding the pickup and dropoff locations, the number of columns in the resulting data matrix is 515.

In [7]:
features = ["PULocationID", "DOLocationID"]
df_yellow_jan_processed.loc[:, features] = df_yellow_jan_processed[features].astype(str)

dv = DictVectorizer()
train_dicts = df_yellow_jan_processed[features].to_dict(orient="records")
X_train = dv.fit_transform(train_dicts)
X_train.shape

/tmp/ipykernel_1812/617100933.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['161' '43' '48' ... '114' '230' '262']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_yellow_jan_processed.loc[:, features] = df_yellow_jan_processed[features].astype(str)
/tmp/ipykernel_1812/617100933.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['141' '237' '238' ... '239' '79' '143']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_yellow_jan_processed.loc[:, features] = df_yellow_jan_processed[features].astype(str)


(3009173, 515)

In [8]:
dv.feature_names_

['DOLocationID=1',
 'DOLocationID=10',
 'DOLocationID=100',
 'DOLocationID=101',
 'DOLocationID=102',
 'DOLocationID=106',
 'DOLocationID=107',
 'DOLocationID=108',
 'DOLocationID=109',
 'DOLocationID=11',
 'DOLocationID=111',
 'DOLocationID=112',
 'DOLocationID=113',
 'DOLocationID=114',
 'DOLocationID=115',
 'DOLocationID=116',
 'DOLocationID=117',
 'DOLocationID=118',
 'DOLocationID=119',
 'DOLocationID=12',
 'DOLocationID=120',
 'DOLocationID=121',
 'DOLocationID=122',
 'DOLocationID=123',
 'DOLocationID=124',
 'DOLocationID=125',
 'DOLocationID=126',
 'DOLocationID=127',
 'DOLocationID=128',
 'DOLocationID=129',
 'DOLocationID=13',
 'DOLocationID=130',
 'DOLocationID=131',
 'DOLocationID=132',
 'DOLocationID=133',
 'DOLocationID=134',
 'DOLocationID=135',
 'DOLocationID=136',
 'DOLocationID=137',
 'DOLocationID=138',
 'DOLocationID=139',
 'DOLocationID=14',
 'DOLocationID=140',
 'DOLocationID=141',
 'DOLocationID=142',
 'DOLocationID=143',
 'DOLocationID=144',
 'DOLocationID=145',

## Question 5: Training a model (B)
A vanilla linear regression model with one-hot encoded versions of pickup and dropoff points as features has a RMSE of 7.64. 

In [9]:
target = "duration"
y_train = df_yellow_jan_processed[target].values

In [10]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_train_pred = lr.predict(X_train)

In [11]:
root_mean_squared_error(y_train, y_train_pred)

7.649261927810898

In [12]:
df_yellow_feb_processed = process_data(df_yellow_feb)
df_yellow_feb_processed = df_yellow_feb_processed[
    (df_yellow_feb_processed["duration"] >= 1)
    & (df_yellow_feb_processed["duration"] <= 60)
]

features = ["PULocationID", "DOLocationID"]
df_yellow_feb_processed.loc[:, features] = df_yellow_feb_processed[features].astype(str)
test_dicts = df_yellow_feb_processed[features].to_dict(orient="records")
X_test = dv.transform(test_dicts)
X_test.shape

/tmp/ipykernel_1812/4228674976.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 1.68333333  0.23333333  0.23333333 ... 14.          7.
  9.8       ]' has dtype incompatible with timedelta64[us], please explicitly cast to a compatible dtype first.
  df.loc[:, "duration"] = df["duration"].dt.total_seconds() / 60.0
/tmp/ipykernel_1812/2225175632.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['142' '132' '161' ... '158' '79' '161']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_yellow_feb_processed.loc[:, features] = df_yellow_feb_processed[features].astype(str)
/tmp/ipykernel_1812/2225175632.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['163' '26' '145' ... '143' '162' '140']' has dtype incompatible with int32, p

(2855951, 515)

In [13]:
y_test_pred = lr.predict(X_test)

target = "duration"
root_mean_squared_error(df_yellow_feb_processed[target].values, y_test_pred)

7.811817853707383